# Analize Image Attributes

In [ ]:
# import os
import random
# import numpy as np
import pandas as pd
# from PIL import Image

import tomllib
import avoddiag as ag

from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns
sns.set_style("dark")

## Load Configuration

In [ ]:
with open("config/config.toml", "rb") as f:
    config = tomllib.load(f)

## Load Image Dataset

In [ ]:
image_dataset = ag.image.data.Dataset(
    root_folder_path="output/TestDataset_Generated_Images"
)

if 'attribute:vehicle_presence' not in image_dataset.metadata['attributes_input']:
    image_dataset.metadata['attributes_input']['attribute:vehicle_presence'] = image_dataset.metadata['attributes_input']['attribute:vehicle_count'] > 0

## Visualize Images

In [ ]:
image_file_name = random.choice(image_dataset.image_file_names)
image_data = image_dataset[image_file_name]

plt.figure(figsize=(8, 8))
plt.imshow(image_data['image'])
plt.axis('off')
plt.title(f"Image: {image_file_name}")

## Compare Input and Generated Attributes

In [ ]:
image_dataset.metadata['attributes_input']

In [ ]:
image_dataset.metadata['attributes_generated:gemini-2.5-flash']

In [ ]:
image_dataset.metadata['attributes_input'][
    [
        'image_file_name',
        'attribute:scene_type',
        'attribute:season',
        'attribute:weather',
        'attribute:vehicle_presence'
    ]
]

In [ ]:
image_dataset.metadata['attributes_generated:gemini-2.5-flash'][
    [
        'image_file_name',
        'attribute:camera_view_confirmation',
        'attribute:scene_type_confirmation',
        'attribute:season',
        'attribute:weather',
        'attribute:vehicle_presence',
    ]
]


### Construct Metadata

In [ ]:
# INPUT METADATA
images_metadata_input = image_dataset.metadata['attributes_input'][
    [
        'image_file_name',
        'attribute:scene_type',
        'attribute:season',
        'attribute:weather',
        'attribute:vehicle_presence'
    ]
].copy()

images_metadata_input.rename(
    columns={
        'attribute:scene_type': 'in:scene',
        'attribute:season': 'in:season',
        'attribute:weather': 'in:weather',
        'attribute:vehicle_presence': 'in:vehicle_presence',
    }, 
    inplace=True
)

images_metadata_input

In [ ]:
# GENERATED METADATA
images_metadata_generated = image_dataset.metadata['attributes_generated:gemini-2.5-flash'][
    [
        'image_file_name',
        'attribute:camera_view_confirmation',
        'attribute:scene_type_confirmation',
        'attribute:scene_type',
        'attribute:scene_concepts',
        'attribute:season_confirmation',
        'attribute:season',
        'attribute:weather_confirmation',
        'attribute:weather',
        'attribute:vehicle_presence',
    ]
].copy()

images_metadata_generated.rename(
    columns={
        'attribute:camera_view_confirmation': 'gen:view_confirmation',
        'attribute:scene_type_confirmation': 'gen:scene_confirmation',
        'attribute:scene_type': 'gen:scene',
        'attribute:scene_concepts': 'gen:scene_concepts',
        'attribute:season_confirmation': 'gen:season_confirmation',
        'attribute:season': 'gen:season',
        'attribute:weather_confirmation': 'gen:weather_confirmation',
        'attribute:weather': 'gen:weather',
        'attribute:vehicle_presence': 'gen:vehicle_presence',
    }, 
    inplace=True
)

for cn in images_metadata_generated.columns:
    if (not cn.startswith('gen')) or (cn.split('_')[-1] != 'confirmation' and cn != 'gen:vehicle_presence') or (images_metadata_generated[cn].dtype == 'bool'):
        continue
    
    assert images_metadata_generated[cn].dtype == 'str', f"Column {cn} is not of type str, but {images_metadata_generated[cn].dtype}"
    
    # print(cn, images_metadata_generated[cn].dtype)
    images_metadata_generated[cn] = images_metadata_generated[cn].apply(lambda x: True if isinstance(x, str) and x.lower() == 'yes' else False)

images_metadata_generated

In [ ]:
# FULL METADATA 
images_metadata = images_metadata_input.merge(images_metadata_generated, on='image_file_name', how='inner')
images_metadata['gen:vehicle_presence_confirmation'] = ~(images_metadata['in:vehicle_presence'] ^ images_metadata['gen:vehicle_presence'])
images_metadata

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

d_plot = images_metadata.rename(columns={
    'gen:view_confirmation': 'Camera View',
    'gen:scene_confirmation': 'Scene Type',
    'gen:season_confirmation': 'Season',
    'gen:weather_confirmation': 'Weather',
}).melt(
    id_vars=['image_file_name'],
    value_vars=[
        'Camera View',
        'Scene Type',
        'Season',
        'Weather',
    ]
)

sns.barplot(
    data=d_plot,
    x='variable',
    y='value',
    width=0.6,
    errorbar=None,
    ax=ax,
)

for p in ax.patches:
    height = p.get_height()
    ax.annotate(f"{height:.1%}", 
                (p.get_x() + p.get_width() / 2, height),
                ha='center', va='bottom', fontsize=10)
    
# ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title("Confirmation Rates per Attribute")
fig.tight_layout()

### Camera View Confirmation

In [ ]:
in_attrib_col_names = [
    'in:scene',
    'in:season',
    'in:weather',
    'in:vehicle_presence',
]

fig, axs = plt.subplots(
    nrows=1, ncols=len(in_attrib_col_names), 
    figsize=(22, 4.5),
    dpi=150
)
axs = axs.flatten()

for (in_attrib_col_name, ax) in zip(in_attrib_col_names, axs):

    d_plot = images_metadata[
        [
            in_attrib_col_name,
            'gen:view_confirmation',
        ]
    ].copy()

    d_plot['gen:view_confirmation'] = d_plot['gen:view_confirmation'].apply(lambda x: 'Approved' if x else 'Rejected')

    in_attrib_col_name_display = in_attrib_col_name.replace('in:', '').replace('_', ' ').title()

    d_plot.rename(
        columns={
            in_attrib_col_name: in_attrib_col_name_display,
            'gen:view_confirmation': 'View Confirmation'
        },
        inplace=True
    )

    d_plot = d_plot.value_counts(sort=False).reset_index(name='count')

    groups  = []
    for i_g, g in d_plot.groupby(in_attrib_col_name_display):
        if g.shape[0] == 2:
            err = (g['count'][g['View Confirmation'] == 'Approved'].iloc[0] - g['count'][g['View Confirmation'] == 'Rejected'].iloc[0]) / g['count'].sum()

        else:
            if g['View Confirmation'].iloc[0] == 'Approved':
                # err = g['count'].iloc[0]
                err = (g['count'].iloc[0]) / g['count'].sum()

            else:
                err = -(g['count'].iloc[0]) / g['count'].sum()

        groups.append(
            (
                g,
                err
            )
        )

    groups = sorted(groups, key=lambda x: x[1], reverse=True)
    d_plot = pd.concat(list(zip(*groups))[0]).reset_index(drop=True)

    sns.barplot(
        data=d_plot,
        x=in_attrib_col_name_display,
        y='count',
        hue='View Confirmation',
        hue_order=['Approved', 'Rejected'],
        # palette=['tab:green', 'tab:red'],
        width=0.6,
        errorbar=None,
        ax=ax
    )

    for container in ax.containers:
        ax.bar_label(
            container, 
            # fmt='%d',
            fmt=lambda x: f"{int(x):,d}",
            # padding=3,
            fontsize=9, 
            rotation=45
        )

    ax.legend(loc='center left')
    ax.set_title(f"by {in_attrib_col_name_display}")
    ax.set_xlabel(None)
    ax.set_ylabel("Number of Images")
    
    # ax.yaxis.get_major_locator().set_params(integer=True)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode="anchor")

    ax.set_ylim(0, d_plot['count'].max() * 1.15)

fig.suptitle("Camera View Confirmation by Input Attribute Values", fontsize=16)
fig.tight_layout()

In [ ]:
# Filter images with correct camera view
images_metadata_view = images_metadata[images_metadata['gen:view_confirmation']].drop(columns=['gen:view_confirmation']).reset_index(drop=True)
assert images_metadata_view[(~images_metadata_view['gen:scene_confirmation']) & (images_metadata_view['in:scene']==images_metadata_view['gen:scene'])].shape[0] == 0, "Inconsistent scene information"
images_metadata_view

### Vehicle Presence Confirmation Analysis

In [ ]:
# ANALYZE THESE IMAGES VISUALLY
images_metadata_view['gen:vehicle_presence_confirmation'].value_counts()

### Attribute Confirmation

In [ ]:
attrib_names = [
    'scene',
    'season',
    'weather',
    'vehicle_presence',
]


# attrib_name_confirm = 'scene'
attrib_name_confirm = 'season'
# attrib_name_confirm = 'weather'
# attrib_name_confirm = 'vehicle_presence'

attrib_confirm_display_name = ' '.join(list(map(lambda x: x.capitalize(), attrib_name_confirm.split('_'))))
attrib_confirm_col_name = f'gen:{attrib_name_confirm}_confirmation'


fig, axs = plt.subplots(
    nrows=1, ncols=len(attrib_names), 
    figsize=(22, 4.5),
    dpi=150
)
axs = axs.flatten()

for (attrib_name, ax) in zip(attrib_names, axs):
    attrib_display_name = ' '.join(list(map(lambda x: x.capitalize(), attrib_name.split('_'))))
    attrib_col_name = f'in:{attrib_name}'

    d_plot = images_metadata_view[
        [
            attrib_col_name,
            attrib_confirm_col_name,
        ]
    ].copy()

    d_plot[attrib_confirm_col_name] = d_plot[attrib_confirm_col_name].apply(lambda x: 'Approved' if x else 'Rejected')

    d_plot.rename(
        columns={
            attrib_col_name: attrib_display_name,
            attrib_confirm_col_name: f'{attrib_confirm_display_name} Confirmation'
        },
        inplace=True
    )

    d_plot = d_plot.value_counts(sort=False).reset_index(name='count')

    groups  = []
    for i_g, g in d_plot.groupby(attrib_display_name):
        if g.shape[0] == 2:
            err = (g['count'][g[f'{attrib_confirm_display_name} Confirmation'] == 'Approved'].iloc[0] - g['count'][g[f'{attrib_confirm_display_name} Confirmation'] == 'Rejected'].iloc[0]) / g['count'].sum()

        else:
            if g[f'{attrib_confirm_display_name} Confirmation'].iloc[0] == 'Approved':
                # err = g['count'].iloc[0]
                err = (g['count'].iloc[0]) / g['count'].sum()

            else:
                err = -(g['count'].iloc[0]) / g['count'].sum()

        groups.append(
            (
                g,
                err
            )
        )

    groups = sorted(groups, key=lambda x: x[1], reverse=True)
    d_plot = pd.concat(list(zip(*groups))[0]).reset_index(drop=True)

    sns.barplot(
        data=d_plot,
        x=attrib_display_name,
        y='count',
        hue=f'{attrib_confirm_display_name} Confirmation',
        hue_order=['Approved', 'Rejected'],
        # palette=['tab:green', 'tab:red'],
        width=0.6,
        errorbar=None,
        ax=ax
    )

    for container in ax.containers:
        ax.bar_label(
            container, 
            # fmt='%d',
            fmt=lambda x: f"{int(x):,d}",
            # padding=3,
            fontsize=9, 
            rotation=45
        )

    ax.legend(loc='center left')
    ax.set_title(f"by {attrib_display_name}")
    ax.set_xlabel(None)
    ax.set_ylabel("Number of Images")
    
    # ax.yaxis.get_major_locator().set_params(integer=True)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode="anchor")

    ax.set_ylim(0, d_plot['count'].max() * 1.15)

fig.suptitle(f"{attrib_confirm_display_name} Confirmation by Input Attribute Values", fontsize=16)
# fig.suptitle(f"{attrib_confirm_display_name} Confirmation by Adjusted Attribute Values", fontsize=16)
fig.tight_layout()

### Adjusting Attribute Values

In [ ]:
from sentence_transformers import SentenceTransformer, util

device = 'cuda:0'

# Load a sentence-transformers model (optimized for semantic similarity)
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

In [ ]:
def category_matching(attrib_name, vc_attrib_in, vc_attrib_gen, model, min_category_image_count, cos_sim_thresh = 0.57):
    while True:
        # Find direct matched between input and generated attribute values based on cosine similarity of text embeddings
        attrib_vals_in = vc_attrib_in[f'in:{attrib_name}'].to_list()
        attrib_vals_gen = vc_attrib_gen[f'gen:{attrib_name}'].to_list()

        embeddings_attrib_vals_in = model.encode(
            attrib_vals_in,
            normalize_embeddings=True
        )
        embeddings_attrib_vals_gen = model.encode(
            attrib_vals_gen,
            normalize_embeddings=True
        )

        cos_sims = util.cos_sim(embeddings_attrib_vals_in, embeddings_attrib_vals_gen)
        cos_sims_max = cos_sims.max(axis=0)
        # print(cos_sims_max)

        vc_attrib_gen['closest_input_attrib_val'] = [attrib_vals_in[i] for i in cos_sims_max.indices]
        vc_attrib_gen['input_cos_sim_max'] = cos_sims_max.values
        vc_attrib_gen['is_matched'] = [True if v >= cos_sim_thresh else False for v in cos_sims_max.values]

        print(
            vc_attrib_gen[vc_attrib_gen['is_matched']]['count'].sum(),
            vc_attrib_gen[vc_attrib_gen['is_matched']]['count'].sum() / vc_attrib_gen['count'].sum()
        )

        # Select a new category to add to the input attribute values
        vc_attrib_gen_unmatched = vc_attrib_gen[~vc_attrib_gen['is_matched']].reset_index(drop=True)
        # assert vc_attrib_gen_unmatched.shape[0] > 0, "No unmatched generated attribute values"
        if vc_attrib_gen_unmatched.shape[0] == 0:
            print(f'No unmatched generated attribute values left. Stopping.')
            break

        attrib_vals_gen_unmatched = vc_attrib_gen_unmatched[f'gen:{attrib_name}'].to_list()
        # attrib_vals_gen_unmatched

        embeddings_attrib_vals_gen_unmatched = model.encode(
            attrib_vals_gen_unmatched,
            normalize_embeddings=True
        )

        cos_sims_unmatched = util.cos_sim(embeddings_attrib_vals_gen_unmatched, embeddings_attrib_vals_gen_unmatched)
        # print(cos_sims_unmatched.shape)

        mask = (cos_sims_unmatched >= cos_sim_thresh)
        assert mask.sum() > 0, "No similar unmatched generated attribute values"
        # print(mask.sum())
        # np.fill_diagonal(mask.numpy(), 0)

        mat = mask.numpy()*vc_attrib_gen_unmatched['count'].to_numpy()

        idx_candidate = mat.sum(axis=1).argmax()
        attribute_val_candidate = attrib_vals_gen_unmatched[idx_candidate]
        num_new_matches_estimated = vc_attrib_gen_unmatched[mask[idx_candidate].numpy()]["count"].sum()

        print(f'New category candidate: "{attribute_val_candidate}" (Expected contribution: {num_new_matches_estimated:,d} new matches)')
        
        if num_new_matches_estimated < min_category_image_count:
            print(f"Stopping criterion reached: Expected contribution {num_new_matches_estimated} < {min_category_image_count}")
            break

        vc_attrib_in.loc[len(vc_attrib_in)] = [attribute_val_candidate, -1]
        # vc_attrib_gen = vc_attrib_gen[vc_attrib_gen['gen:scene'] != attribute_val_candidate].reset_index(drop=True)
        min_category_image_count = 0.5*4000/vc_attrib_in.shape[0] # TODO: Try to understand this
        print()

In [ ]:
images_metadata_view

In [ ]:
attribute_names = ['scene', 'season', 'weather']

for attrib_name in attribute_names:
    print(f'Processing attribute: "{attrib_name}"')

    d_unconfirmed = images_metadata_view[~images_metadata_view[f'gen:{attrib_name}_confirmation']][
        [
            'image_file_name',
            f'in:{attrib_name}',
            f'gen:{attrib_name}',
        ]
    ].reset_index(drop=True)

    vc_attrib_in = images_metadata[f'in:{attrib_name}'].value_counts().to_frame().merge(
        d_unconfirmed[f'in:{attrib_name}'].value_counts().to_frame(), 
        left_index=True, 
        right_index=True, 
        how='left', 
        suffixes=('_total', '')
    ).fillna(0).astype(int).drop(columns=['count_total']).sort_values('count', ascending=False).reset_index(drop=False)
    
    vc_attrib_gen = d_unconfirmed[f'gen:{attrib_name}'].value_counts().to_frame().reset_index()

    print(vc_attrib_in.shape[0], vc_attrib_gen.shape[0])

    
    if (vc_attrib_in.shape[0] > 0) and (vc_attrib_gen.shape[0] > 0):
        category_matching(
            attrib_name,
            vc_attrib_in, 
            vc_attrib_gen, 
            model, 
            min_category_image_count=0.5*4000/vc_attrib_in.shape[0], # Weather
            cos_sim_thresh=0.57
        )

        category_mapping = dict(
            vc_attrib_gen[vc_attrib_gen['is_matched']][
                [f'gen:{attrib_name}', 'closest_input_attrib_val']
            ].apply(lambda x: (x[f'gen:{attrib_name}'], x['closest_input_attrib_val']), axis=1).to_list()
        )

    elif (vc_attrib_in.shape[0] > 0) and (vc_attrib_gen.shape[0] == 0):
        category_mapping = dict()

    else:
        raise NotImplementedError()


    images_metadata_view[f'gen:{attrib_name}_adjusted'] = images_metadata_view.apply(
        lambda row: row[f'in:{attrib_name}'] if row[f'gen:{attrib_name}_confirmation'] else category_mapping.get(row[f'gen:{attrib_name}'], 'N/A'),
        axis=1
    )


In [ ]:
images_metadata_view

### Filter Images Based on the relabeling process

In [ ]:
# Count how many images have unlabeled attributes after adjustment
# These needs to be removed from the dataset
mask_None = images_metadata_view[
    [
        'gen:scene_adjusted',
        'gen:season_adjusted',
        'gen:weather_adjusted',
    ]
].notna().all(axis=1)

mask_NotNA = (images_metadata_view[
    [
        'gen:scene_adjusted',
        'gen:season_adjusted',
        'gen:weather_adjusted',
    ]
] != 'N/A').all(axis=1)

mask_vehicle_presence_confirmed = images_metadata_view['gen:vehicle_presence_confirmation']

mask_filter = mask_None & mask_vehicle_presence_confirmed
# maskmask_filter = mask_None & mask_vehicle_presence_confirmed & mask_NotNA

print(f'Number of images with all attributes labeled after adjustment:  {mask_None.sum():,d} ({(~mask_None).sum(): >5,d})')
print(f'Number of images with confirmed vehicle presence:               {mask_vehicle_presence_confirmed.sum():,d} ({(~mask_vehicle_presence_confirmed).sum(): >5,d})')
print(f'Number of images with any attribute not labeled "N/A":          {mask_NotNA.sum():,d} ({(~mask_NotNA).sum(): >5,d})')

print()
print(f'Number of images with all filters applied:                      {mask_filter.sum(): >5,d} ({(~mask_filter).sum(): >5,d})')

In [ ]:
images_metadata_filtered = images_metadata_view[mask_filter].reset_index(drop=True)
images_metadata_filtered

### Plot Final Attribute Distribution

In [ ]:
# Plot final attribute distributions
attrib_names = [
    'scene',
    'season',
    'weather',
    'vehicle_presence',
]

fig, axs = plt.subplots(
    nrows=2, ncols=len(attrib_names), 
    figsize=(22, 9),
    # dpi=150
)
# axs = axs.flatten()

for i, attrib_name in enumerate(attrib_names):
    attrib_display_name = ' '.join(list(map(lambda x: x.capitalize(), attrib_name.split('_'))))

    

    if attrib_name == 'vehicle_presence':
        attrib_col_name = f'gen:{attrib_name}'
    else:
        attrib_col_name = f'gen:{attrib_name}_adjusted'

    d_plot = images_metadata_filtered[
        [
            attrib_col_name,
        ]
    ].copy()

    d_plot.rename(
        columns={
            attrib_col_name: attrib_display_name,
        },
        inplace=True
    )

    d_plot = d_plot.value_counts(sort=False).reset_index(name='count')
    d_plot['count_norm'] = d_plot['count'] / d_plot['count'].sum()
    # d_plot = d_plot.sort_values(by='count', ascending=False).reset_index(drop=True)

    ax = axs[0, i]

    sns.barplot(
        data=d_plot,
        x=attrib_display_name,
        y='count',
        width=0.6,
        errorbar=None,
        ax=ax
    )

    for container in ax.containers:
        ax.bar_label(
            container, 
            # fmt='%d',
            fmt=lambda x: f"{int(x):,d}",
            # padding=3,
            fontsize=9, 
            rotation=45
        )

    # Change color of "N/A" category to red
    # for bar, label in zip(ax.patches, d_plot[attrib_display_name]):
    #     if label == "N/A":
    #         # bar.set_color("gray")
    #         bar.set_alpha(0.5)

    for label in ax.get_xmajorticklabels():
        if label.get_text() == "N/A":
            # label.set_color("red")
            # label.set_fontweight("bold")
            label.set_bbox(dict(facecolor='red', alpha=0.3, edgecolor='none'))

    ax.set_title(f"Final {attrib_display_name} Distribution")
    ax.set_xlabel(None)
    ax.set_ylabel("Number of Images")
    
    # ax.yaxis.get_major_locator().set_params(integer=True)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode="anchor")

    ax.set_ylim(0, d_plot['count'].max() * 1.15)

    ax.axhline(y=d_plot['count'].max(), color='r', linestyle='--', label='Mean')

    # Normalized
    ax = axs[1, i]

    sns.barplot(
        data=d_plot,
        x=attrib_display_name,
        y='count_norm',
        width=0.6,
        errorbar=None,
        ax=ax
    )

    for container in ax.containers:
        ax.bar_label(
            container, 
            # fmt='%d',
            fmt=lambda x: f"{x:.1%}",
            # padding=3,
            fontsize=9, 
            rotation=45
        )

    for label in ax.get_xmajorticklabels():
        if label.get_text() == "N/A":
            # label.set_color("red")
            # label.set_fontweight("bold")
            label.set_bbox(dict(facecolor='red', alpha=0.3, edgecolor='none'))

    ax.set_title(f"Final {attrib_display_name} Distribution")
    ax.set_xlabel(None)
    ax.set_ylabel("Number of Images")
    
    # ax.yaxis.get_major_locator().set_params(integer=True)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode="anchor")

    ax.set_ylim(0, d_plot['count_norm'].max() * 1.15)

    ax.axhline(y=d_plot['count_norm'].max(), color='r', linestyle='--', label='Mean')

    ax.yaxis.set_major_formatter(PercentFormatter(1.0))


fig.suptitle(
    f"Final Attribute Distributions after Filtering",
    # fontsize=16
)
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(
    nrows=2, ncols=len(attrib_names), 
    figsize=(22, 9),
    # dpi=150
)

for i, attrib_name in enumerate(attrib_names):
    attrib_col_name = f'in:{attrib_name}'
    attrib_display_name = ' '.join(list(map(lambda x: x.capitalize(), attrib_name.split('_'))))

    print(f"{attrib_col_name}: {images_metadata_filtered[attrib_col_name].nunique()} unique values")

    d_plot = images_metadata[attrib_col_name].value_counts(sort=False).reset_index(name='count').sort_values(by=attrib_col_name, ascending=False).reset_index(drop=True)
    d_plot['count_norm'] = d_plot['count'] / d_plot['count'].sum()

    # ABSOLUTE VALS
    ax = axs[0, i]
    sns.barplot(
        data=d_plot, 
        x=attrib_col_name, 
        y='count',
        ax=ax
    )

    for container in ax.containers:
        ax.bar_label(
            container, 
            # fmt='%d',
            fmt=lambda x: f"{int(x):,d}",
            # padding=3,
            fontsize=9, 
            rotation=45
        )
    
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode="anchor")
    ax.set_ylim(0, d_plot['count'].max() * 1.15)
    ax.axhline(y=d_plot['count'].max(), color='r', linestyle='--', label='Mean')

    ax.set_xlabel(None)
    ax.set_ylabel("Number of Images")

    # NORMALIZED VALS
    ax = axs[1, i]
    sns.barplot(
        data=d_plot, 
        x=attrib_col_name, 
        y='count_norm',
        ax=ax
    )

    for container in ax.containers:
        ax.bar_label(
            container, 
            # fmt='%d',
            fmt=lambda x: f"{x:.1%}",
            # padding=3,
            fontsize=9, 
            rotation=45
        )

    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode="anchor")
    ax.set_ylim(0, d_plot['count_norm'].max() * 1.15)
    ax.axhline(y=d_plot['count_norm'].max(), color='r', linestyle='--', label='Mean')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    
    ax.set_xlabel(None)
    ax.set_ylabel("Normalized Number of Images")

    # break

fig.set_tight_layout(True)

In [ ]:
# Plot final attribute distributions
attrib_names = [
    'scene',
    'season',
    'weather',
    'vehicle_presence',
]

fig, axs = plt.subplots(
    nrows=2, ncols=len(attrib_names), 
    figsize=(22, 9),
    # dpi=150
)
# axs = axs.flatten()

for i, attrib_name in enumerate(attrib_names):
    attrib_display_name = ' '.join(list(map(lambda x: x.capitalize(), attrib_name.split('_'))))

    attrib_col_name_in = f'in:{attrib_name}'
    if attrib_name == 'vehicle_presence':
        attrib_col_name_gen = f'gen:{attrib_name}'
    else:
        attrib_col_name_gen = f'gen:{attrib_name}_adjusted'

    d_plot_in = images_metadata[attrib_col_name_in].value_counts(sort=False).reset_index(name='count').sort_values(by=attrib_col_name_in).reset_index(drop=True)
    d_plot_in['count_norm'] = d_plot_in['count'] / d_plot_in['count'].sum()
    d_plot_in.rename(
        columns={
            attrib_col_name_in: attrib_display_name
        },
        inplace=True
    )

    d_plot_gen_filtered = images_metadata_filtered[attrib_col_name_gen].value_counts(sort=False).reset_index(name='count').sort_values(by=attrib_col_name_gen).reset_index(drop=True)
    d_plot_gen_filtered['count_norm'] = d_plot_gen_filtered['count'] / d_plot_gen_filtered['count'].sum()
    d_plot_gen_filtered.rename(
        columns={
            attrib_col_name_gen: attrib_display_name
        },
        inplace=True
    )

    d_plot_in['type'] = 'Input'
    d_plot_gen_filtered['type'] = 'Generated (Filtered)'
    d_plot = pd.concat([d_plot_in, d_plot_gen_filtered], ignore_index=True) 
    
    ax = axs[0, i]

    sns.barplot(
        data=d_plot,
        x=attrib_display_name,
        y='count',
        hue='type',
        hue_order=['Input', 'Generated (Filtered)'],
        # palette=['tab:blue', 'tab:orange'],
        width=0.6,
        errorbar=None,
        ax=ax
    )
    
    for container in ax.containers:
        ax.bar_label(
            container, 
            # fmt='%d',
            fmt=lambda x: f"{int(x):,d}",
            # padding=3,
            fontsize=9, 
            rotation=45
        )

    # Change color of "N/A" category to red
    # for bar, label in zip(ax.patches, d_plot[attrib_display_name]):
    #     if label == "N/A":
    #         # bar.set_color("gray")
    #         bar.set_alpha(0.5)

    for label in ax.get_xmajorticklabels():
        if label.get_text() == "N/A":
            # label.set_color("red")
            # label.set_fontweight("bold")
            label.set_bbox(dict(facecolor='red', alpha=0.3, edgecolor='none'))

    ax.set_title(f"Final {attrib_display_name} Distribution")
    ax.set_xlabel(None)
    ax.set_ylabel("Number of Images")
    
    # ax.yaxis.get_major_locator().set_params(integer=True)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode="anchor")

    ax.set_ylim(0, d_plot['count'].max() * 1.15)

    ax.axhline(y=d_plot['count'].max(), color='r', linestyle='--', label='Mean')

    # Normalized
    ax = axs[1, i]

    sns.barplot(
        data=d_plot,
        x=attrib_display_name,
        y='count_norm',
        hue='type',
        hue_order=['Input', 'Generated (Filtered)'],
        # palette=['tab:blue', 'tab:orange'],
        width=0.6,
        errorbar=None,
        ax=ax
    )

    for container in ax.containers:
        ax.bar_label(
            container, 
            # fmt='%d',
            fmt=lambda x: f"{x:.1%}",
            # padding=3,
            fontsize=9, 
            rotation=45
        )

    for label in ax.get_xmajorticklabels():
        if label.get_text() == "N/A":
            # label.set_color("red")
            # label.set_fontweight("bold")
            label.set_bbox(dict(facecolor='red', alpha=0.3, edgecolor='none'))

    ax.set_title(f"Final {attrib_display_name} Distribution")
    ax.set_xlabel(None)
    ax.set_ylabel("Number of Images")
    
    # ax.yaxis.get_major_locator().set_params(integer=True)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode="anchor")

    ax.set_ylim(0, d_plot['count_norm'].max() * 1.15)

    ax.axhline(y=d_plot['count_norm'].max(), color='r', linestyle='--', label='Mean')

    ax.yaxis.set_major_formatter(PercentFormatter(1.0))


fig.suptitle(
    f"Final Attribute Distributions after Filtering",
    # fontsize=16
)
fig.tight_layout()

In [ ]:
# Plot final attribute distributions
attrib_names = [
    'scene',
    'season',
    'weather',
    'vehicle_presence',
]

fig, axs = plt.subplots(
    nrows=1, ncols=len(attrib_names), 
    figsize=(22, 4.5),
    # dpi=150
)
axs = axs.flatten()

for attrib_name, ax in zip(attrib_names, axs):
    attrib_display_name = ' '.join(list(map(lambda x: x.capitalize(), attrib_name.split('_'))))

    attrib_col_name_in = f'in:{attrib_name}'
    if attrib_name == 'vehicle_presence':
        attrib_col_name_gen = f'gen:{attrib_name}'
    else:
        attrib_col_name_gen = f'gen:{attrib_name}_adjusted'

    d_plot_in = images_metadata[attrib_col_name_in].value_counts(sort=False).reset_index(name='count').sort_values(by=attrib_col_name_in).reset_index(drop=True)
    d_plot_in['count_norm'] = d_plot_in['count'] / d_plot_in['count'].sum()
    d_plot_in.rename(
        columns={
            attrib_col_name_in: attrib_display_name
        },
        inplace=True
    )

    d_plot_gen_filtered = images_metadata_filtered[attrib_col_name_gen].value_counts(sort=False).reset_index(name='count').sort_values(by=attrib_col_name_gen).reset_index(drop=True)
    d_plot_gen_filtered['count_norm'] = d_plot_gen_filtered['count'] / d_plot_gen_filtered['count'].sum()
    d_plot_gen_filtered.rename(
        columns={
            attrib_col_name_gen: attrib_display_name
        },
        inplace=True
    )

    d_plot_in['type'] = 'Input'
    d_plot_gen_filtered['type'] = 'Generated (Filtered)'
    d_plot = pd.concat([d_plot_in, d_plot_gen_filtered], ignore_index=True) 

    sns.barplot(
        data=d_plot,
        x=attrib_display_name,
        y='count',
        hue='type',
        hue_order=['Input', 'Generated (Filtered)'],
        # palette=['tab:blue', 'tab:orange'],
        width=0.6,
        errorbar=None,
        ax=ax
    )
    
    for container in ax.containers:
        ax.bar_label(
            container, 
            # fmt='%d',
            fmt=lambda x: f"{int(x):,d}",
            # padding=3,
            fontsize=9, 
            rotation=45
        )

    # Change color of "N/A" category to red
    # for bar, label in zip(ax.patches, d_plot[attrib_display_name]):
    #     if label == "N/A":
    #         # bar.set_color("gray")
    #         bar.set_alpha(0.5)

    for label in ax.get_xmajorticklabels():
        if label.get_text() == "N/A":
            # label.set_color("red")
            # label.set_fontweight("bold")
            label.set_bbox(dict(facecolor='red', alpha=0.3, edgecolor='none'))

    ax.set_title(f"Final {attrib_display_name} Distribution")
    ax.set_xlabel(None)
    ax.set_ylabel("Number of Images")
    
    # ax.yaxis.get_major_locator().set_params(integer=True)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode="anchor")

    ax.set_ylim(0, d_plot['count'].max() * 1.15)

    ax.axhline(y=d_plot['count'].max(), color='r', linestyle='--', alpha=0.5)

    ax.legend(loc='lower center', ncol=2)


fig.suptitle(
    f"Attribute Distributions of the Generated Images Before and After Filtering",
    # fontsize=16
)
fig.tight_layout()

### Store Metadata

In [ ]:
image_dataset.metadata.keys()

In [ ]:
metadata_key = 'attributes_filtered:gemini-2.5-flash'

In [ ]:
# Store the filtered metadata
image_dataset.metadata[metadata_key] = images_metadata_filtered
image_dataset.save_metadata(keys_to_overwrite=[metadata_key])

In [ ]:
# 
images_metadata_filtered = image_dataset.metadata[metadata_key]
images_metadata_filtered